# Setting parameters

In this tutorial, we will explain how to set the different parameters using the SetupParams class. This step is crucial in order to use this python module.

#### Plan of this notebook
1) Brief introduction to the different parameters
2) How to set the parameters: fixing the trigger rate, the non-vacuum probability or the multi-photon probability
3) Calculating the SNR, the signal rate and the noise rate

After this short tutorial, you will be able to initialize the SetupParams class, set the different parameters and compute the signal rate, the noise rate and the SNR of different sources. With that knowledge, you will be able to go to the other tutorials that are provided. 

## 1) Brief introduction to the different parameters

The module aimed at modelling quantum lidars based on three different sources:
- Attenuated pulse laser : coherent state
- Single photon source (SPS) using quantum dot
- Entangled photon source (EPS) using SPDC : squeezed state

In order to model a quantum lidar correctly based on these sources, we need to set the different parameters. Here is the list of the parameters that can be set: 

### Source parameters

- `output_power` : Optical power of the source. $[s^{-1}]$
- `multi_photon_probability` : The multi-photon probability of the source. $[-]$
- `no_vacuum_probability` : The non-vacuum probability of the source. $[-]$
- `sp_collection` : The collection efficiency of the source. $[-]$
- `sp_p1` : Probability that the SPS emits a single photon. $[-]$
- `sp_p2` : Probability that the SPS emits two photons. $[-]$
- `spdc_eps_heralding` : The idler collection efficiency. $[-]$
- `spdc_eps_collection` : The collection efficiency of the EPS. $[-]$

### Detection parameters

- `atmosphere` : The atmospheric transmission efficiency. $[dB/km]$
- `target_distance` : The distance between the source and the target. $[m]$
- `receiver_diameter` : The diameter of the receiver. $[m]$
- `target_albedo` : The albedo of the target. $[-]$
- `optics_transmitter` : The transmission efficiency of the optical system. $[-]$
- `optics_receiver` : The reception efficiency of the optical system. $[-]$
- `detection_efficiency` : The detection efficiency of the detector. $[-]$
- `background` : The background noise detection. $[Hz/cm^2]$
- `detector_dark` : The dark count of the detector. $[Hz]$
- `timing_window` : The integration time of the detector. $[s]$

In the model that we use for the model, we use the variable $\eta_{signal}$ for the collection efficiency. When it comes to the implementation, a distinction is made between the collection efficiency of the SPS and the EPS.

In [1]:
from Sources import SetupParameters, PulsedLaser, EntangledPhotonSPDC, SinglePhoton

After importing the SetupParameters class, one can use the function help() to get a brief description of the class.

In [2]:
SetupParameters.help()

Here are the parameters you can set:
--- General ---
fock_space_dim: Dimension of the fock space, must be an integer
--- Sources ---
output_power: optical output power of the source in photon per second [s^-1]
multi_photon_probability: Probability of the laser source emitting more than 1 photon [-]
sp_collection: Efficiency of the single photon source collection, does not include detector efficiency [-]
sp_p1: Probability that the single photon source emits 1 photons [-]
sp_p2: Probability that the single photon source emits 2 photons [-]
spdc_eps_heralding: Efficiency of the EPS local measurement, includes collection and detector efficiency [-]
spdc_eps_collection: Efficiency of the EPS collection for the sensing photon, does not include detector efficiency [-]
spdc_emission: SPDC average pair photon number [-]
--- Detection ---
target_distance: Distance to target, loss is assumed as a 2*pi sphere scattering from target, no other channel losses included
atmosphere: Efficiency of propa

## 2) How to set the parameters

Simply put, the SetupParameters act as a dictionary where the different parameters can be added. However, the two following parameters are different in the sense that only one of them can be fixed:
- Triggering rate
- Non-vacuum probability
- Multi-photon probability

In the code, you must specify one of these three values and leave the other one at `None`, otherwise, an error will be raised.

#### Example : setup parameters for a laser

Now, let's define the parameters for an attenuated laser. Since we are not interested with the other sources for now, we can leave the different collection efficiencies at `None`.

Let's fix the multi-photon probability at 5%. Since the multi-photon probability is fixed, we must leave the non-vacuum probability at `None`.

In [14]:
param_laser = SetupParameters(
	fock_space_dim=5,
	output_power=2e6,
	trigger_rate=None,
	multi_photon_probability=0.05,
	no_vacuum_probability=None,
	sp_collection=None,
	sp_p1=None,
	sp_p2=None,
	spdc_eps_heralding=None,
	spdc_eps_collection=None,
	atmosphere=1,
	target_distance=1,
	receiver_diameter=0.05,
	target_albedo=0.2,
	optics_transmitter=0.8,
	optics_receiver=0.5,
	detection_efficiency=0.5,
	background=20,
	detector_dark=200,
	timing_window=0.5e-9,
)

Now, let's pass the parameters to the PulsedLaser class and let's exact the different probabilities.

In [15]:
laser = PulsedLaser(param_laser)

print(f"The Non-vacuum probability of the laser is {laser.no_vacuum_probability*100:.0f}%")
print(f"The multi-photon probability of the laser is {laser.multi_photon_probability*100:.0f}%")

The Non-vacuum probability of the laser is 30%
The multi-photon probability of the laser is 5%


#### Example : setup parameters for an EPS
As expected, the multi-photon probability is 5% and the other probabilities are computed accordingly. Now, we can do something similar for the EPS. In general, it is recommended (but not mandatory) to perform a deepcopy of the parameters in order to avoid any conflict between the different sources.

In [16]:
from copy import deepcopy

param_eps = deepcopy(param_laser)
param_eps["spdc_eps_heralding"] = 0.1
param_eps["spdc_eps_collection"] = 0.2
param_eps["multi_photon_probability"] = None

Now, let's fix the non-vacuum probability of the EPS at 30%.

In [17]:
param_eps["no_vacuum_probability"] = 0.3

eps = EntangledPhotonSPDC(param_eps)

print(f"The Non-vacuum probability of the EPS is {eps.no_vacuum_probability*100:.0f}%")
print(f"The multi-photon probability of the EPS is {eps.multi_photon_probability*100:.0f}%")

The Non-vacuum probability of the EPS is 30%
The multi-photon probability of the EPS is 9%


#### Example : setup parameters for a SPS

The SPS is a little bit trickier because its average photon per pulse is fixed by its purity. Hence, once the optical power is fixed, then the the non-vacuum probability and the multi-photon probability are fixed. The easiest way to deal with the SPS is therefore to simply leave those three parameters at `None`, and the code will do the rest. You can still fix one of the three parameters, but it most be the expected value, otherwise an error will be raised.

In [18]:
param_sps = deepcopy(param_laser)
param_sps["sp_collection"] = 0.2
param_sps["sp_p1"] = 0.99
param_sps["sp_p2"] = 0.001
param_sps["multi_photon_probability"] = None

sps = SinglePhoton(param_sps)

print(f"The Non-vacuum probability of the SPS is {sps.no_vacuum_probability*100:.0f}%")
print(f"The multi-photon probability of the SPS is {sps.multi_photon_probability*100:.4f}%")

The Non-vacuum probability of the SPS is 16%
The multi-photon probability of the SPS is 0.0026%


## 3) Calculating the SNR, the signal rate and the noise rate

Once the parameters are set, one can calculate the SNR, the signal rate $\mathcal{C}$ and the noise rate $\mathcal{C}_{noise}$. The SNR is defined as follows:

$$\textbf{SNR} = \frac{\mathcal{C}-\mathcal{C}_{noise}}{\mathcal{C}_{noise}}$$

The signal rate is the number of photons that are detected per second in the bin associated with the time-of-flight of the target. This includes the photon that are coming back from the target as well as the background noise. The noise rate is the number of photons that are detected per second in the other bins. It is important to remember that these two rates are the rate for a bin with a time window $T_w$

#### Example : Compute the signal rate of all sources

In [19]:
param = SetupParameters(
	fock_space_dim=5,
	output_power=2e6,
	trigger_rate=None,
	multi_photon_probability=None,
	no_vacuum_probability=None,
	sp_collection=0.8,
	sp_p1=0.99,
	sp_p2=1e-3,
	spdc_eps_heralding=0.4,
	spdc_eps_collection=0.8,
	atmosphere=1,
	target_distance=1,
	receiver_diameter=0.05,
	target_albedo=0.2,
	optics_transmitter=0.8,
	optics_receiver=0.5,
	detection_efficiency=0.5,
	background=20,
	detector_dark=200,
	timing_window=0.5e-9,
)

sps = SinglePhoton(param)
multi_photon_probability = sps.multi_photon_probability
param["multi_photon_probability"] = multi_photon_probability

laser = PulsedLaser(param)
eps = EntangledPhotonSPDC(param)

signal_laser = laser.signal_rate()
signal_eps = eps.signal_rate()
signal_sps = sps.signal_rate()

print(f"The signal rate of the laser is {signal_laser:.2f} photons/s")
print(f"The signal rate of the EPS is {signal_eps:.2f} photons/s")
print(f"The signal rate of the SPS is {signal_sps:.2f} photons/s")

The signal rate of the laser is 51.75 photons/s
The signal rate of the EPS is 13.33 photons/s
The signal rate of the SPS is 32.17 photons/s


#### Example : Compute the noise rate of all sources

In [20]:
noise_laser = laser.noise_rate()
noise_eps = eps.noise_rate()
noise_sps = sps.noise_rate()

print(f"The noise rate of the laser is {noise_laser:.2f} photons/s")
print(f"The noise rate of the EPS is {noise_eps:.2f} photons/s")
print(f"The noise rate of the SPS is {noise_sps:.2f} photons/s")

The noise rate of the laser is 20.51 photons/s
The noise rate of the EPS is 0.37 photons/s
The noise rate of the SPS is 0.93 photons/s


#### Example : Compute the SNR of all sources

In [21]:
snr_laser = laser.signal_to_noise_rate()
snr_eps = eps.signal_to_noise_rate()
snr_sps = sps.signal_to_noise_rate()

print(f"The SNR of the laser is {snr_laser:.2f}")
print(f"The SNR of the EPS is {snr_eps:.2f}")
print(f"The SNR of the SPS is {snr_sps:.2f}")

The SNR of the laser is 1.52
The SNR of the EPS is 35.46
The SNR of the SPS is 33.46


In this tutorial, we have seen how to set the different parameters using the SetupParameters class. We have also seen how to compute the signal rate, the noise rate and the SNR. With that knowledge, you will be able to go to the other tutorials that are provided.
